# Hafta 5 — Colab kurulum ve uçtan uca doğrulama

Bu defter paket kurulumunu, testleri ve örnek dosyaları doğrular. Model eğitimi veya 216 kayıtlık deneyleri başlatmaz. Sonuçlar `Hafta5/colab_dogrulama/` içine yazılır; Hafta4 ölçümlerini değiştirmez.

**Kanıt ayrımı:** 4 Kasım 2026'da GitHub main `3018b19` sürümünde 128 test geçti, referans RTTM klasörü olmadığı için bir test atlandı ve `KURULUM: OK` alındı. İlk duman testinde kısa/sessiz dosyaların dört satırı Errno 95 verdi. Ayrı süreçlerde tekrar edilen dört satırın tamamı geçti. İlk hatanın teknik nedeni kesinleşmedi; kalıcı hata düzeltmesi yapıldığı iddia edilmez. Bu yeni defterin kendi Colab çalışması ayrıca doğrulanmalıdır.

## 0. Ayarlar
Drive'da paylaşılan proje klasörünü Drive'ım bölümüne kısayol olarak ekleyin. CPU yeterlidir. Dal, PR birleştirildikten sonra `main` yapılabilir; çalıştırılan commit çıktıda kaydedilir.


In [ ]:
import os, sys, json, subprocess, shutil, time
from pathlib import Path
from google.colab import drive

KOK = Path('/content/drive/MyDrive/BilgisayarTasarim_Proje68')
DAL = 'cevat-can/hafta5-dogrulama'
REPO = Path('/content/Konusmaci-Tespiti-hafta5')
SONUC = KOK / 'Hafta5/colab_dogrulama'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['KT_MODEL_ONBELLEK'] = str(KOK / 'Hafta4/modeller/ecapa')


## 1. Kurulum
Bu defter mevcut Colab torch/torchaudio sürümlerini korur. Python ve tüm paket sürümleri kaydedilir. Kurulumda hata varsa aşağıdaki testlere geçmeyin. Paketler değişince oturumu yeniden başlatıp 0. bölümü tekrar çalıştırın; oturumu silmeyin.


In [ ]:
drive.mount('/content/drive')
if not KOK.is_dir():
    raise FileNotFoundError(f'Proje klasörü bulunamadı: {KOK}; Drive kısayolunu kontrol edin')
SONUC.mkdir(parents=True, exist_ok=True)
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', DAL, 'https://github.com/HekimSefkan/Konusmaci-Tespiti.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', DAL], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', DAL], check=True)
    subprocess.run(['git', '-C', str(REPO), 'merge', '--ff-only', f'origin/{DAL}'], check=True)
print('Commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())
secilen = [s.strip() for s in (REPO / 'requirements.txt').read_text().splitlines()
           if s.strip() and not s.lstrip().startswith(('#', '--extra-index-url', 'torch==', 'torchaudio=='))]
gereksinim = Path('/content/hafta5_gereksinimler_colab.txt')
gereksinim.write_text('\n'.join(secilen)+'\n')
p = subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(gereksinim)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
(SONUC / 'pip_kurulum.log').write_text(p.stdout)
print(p.stdout)
if p.returncode:
    raise RuntimeError('Paket kurulumu başarısız; pip_kurulum.log dosyasındaki asıl hatayı inceleyin')
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(SONUC / 'paket_surumleri.txt').write_text(freeze)
(SONUC / 'python_commit.txt').write_text(sys.version + '\n' + subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True))
print('Kurulum tamamlandı. Paket sürümleri değiştiyse oturumu yeniden başlatıp 0. bölümü çalıştırın.')


### Yalnızca `numpy` için `uninstall-no-record-file` hatası alınırsa
4 Kasım 2026 Colab oturumunda NumPy 1.26.4 için RECORD kaydı eksikti. Bu hata olursa aşağıdaki onarım ayrı bir hücrede çalıştırılıp 1. bölüm yeniden denenebilir. Genel bir sürüm çatışması çözümü değildir; otomatik uygulanmaz. `--ignore-installed` mevcut paketi kaldırmadan üzerine kurar; yalnız geçici Colab ortamında ve bu belirli hata için kullanın.

```python
subprocess.run([sys.executable, '-m', 'pip', 'install', '--ignore-installed', '--no-deps', 'numpy==2.2.6'], check=True)
```

Paket kurulumu başarılı tamamlandıktan sonra oturumu yeniden başlatın. Pip seçeneği: https://pip.pypa.io/en/stable/cli/pip_install/#cmdoption-I

## 2. Testler
Referans klasörü yoksa ilgili veri bölme testi gerekçesiyle atlanabilir. Bu atlamayı raporda tam test başarısı diye göstermeyin.


In [ ]:
for ad, komut in [
    ('pytest', [sys.executable, '-m', 'pytest', '-q', '-rs', '-p', 'no:cacheprovider', 'tests']),
    ('kurulum', [sys.executable, 'tests/kurulum_kontrol.py']),
]:
    p = subprocess.run(komut, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    (SONUC / f'{ad}.log').write_text(p.stdout)
    if p.returncode:
        raise RuntimeError(f'{ad} başarısız: {p.returncode}')


## 3. Duman testi
11 dosya iki yöntemle ayrı Python süreçlerinde çalıştırılır. Çıkış kodu yanında sessiz/kısa dosyanın RTTM içeriği de kontrol edilir. İlk başarısız deneme korunur. Tekrar için aşağıdaki döngüyü yalnız başarısız satırların dosya adlarıyla çalıştırın ve farklı CSV adı kullanın; ilk kaydı başarıyla değiştirmeyin. Errno 95 tekrar görülürse tam stderr günlüğü kaydedilir.


In [ ]:
import csv
sesler = sorted((KOK / 'Hafta2/ornek_ses').glob('*.wav'))
sesler += sorted(p for p in (KOK / 'Hafta2/ornek_ses/edge').iterdir() if p.is_file())
if len(sesler) != 11:
    raise ValueError(f'Duman testi 11 dosya gerektiriyor; bulunan {len(sesler)}')
calisma = SONUC / time.strftime('duman_%Y%m%d_%H%M%S')
calisma.mkdir(parents=True)
satirlar = []
for yontem in ('mfcc', 'ecapa'):
    for ses in sesler:
        hyp = calisma / f'{yontem}_{ses.stem}.rttm'
        p = subprocess.run([sys.executable, '-m', 'src.main', str(ses), '-o', str(hyp), '--yontem', yontem], cwd=REPO, capture_output=True, text=True)
        (calisma / f'{yontem}_{ses.stem}.log').write_text(p.stdout+'\n'+p.stderr)
        beklenen = 1 if ses.stem in ('bozuk_yarim', 'desteklenmeyen_bicim') else 0
        ok = p.returncode == beklenen
        if beklenen == 0:
            ok = ok and hyp.is_file()
        if ok and ses.stem in ('sessiz_5sn', 'kisa_1sn'):
            kontrol = "from src.cikti import rttm_oku; import sys; s=rttm_oku(sys.argv[1]); assert (not s if sys.argv[2]=='sessiz_5sn' else len({x[2] for x in s})<=1)"
            ok = subprocess.run([sys.executable, '-c', kontrol, str(hyp), ses.stem], cwd=REPO).returncode == 0
        satirlar.append({'yontem':yontem, 'dosya':ses.name, 'cikis':p.returncode, 'beklenti':'OK' if ok else 'HATA'})
        print(yontem, ses.name, 'çıkış', p.returncode, satirlar[-1]['beklenti'], flush=True)
with (calisma / 'duman.csv').open('w', newline='') as f:
    w=csv.DictWriter(f, fieldnames=['yontem','dosya','cikis','beklenti']); w.writeheader(); w.writerows(satirlar)
print('Duman testi:', 'GEÇTİ' if satirlar and all(s['beklenti']=='OK' for s in satirlar) else 'BEKLENTİYE UYMAYAN SATIR VAR')
print('Günlük klasörü:', calisma)
